# LightGuard — Train in Colab

**Runtime:** `Runtime → Change runtime type → CPU` and select **High-RAM** if available.  
GPU is not needed — LightGBM runs on CPU.  High-RAM gives more disk quota (~200 GB vs ~100 GB).

---

## What this notebook does

1. Install dependencies (thrember + LightGuard requirements).
2. Clone the LightGuard repo and import `src/lightguard`.
3. Load `config.yaml` — all paths and hyperparameters come from there.
4. Download the EMBER2024 PE file types listed in `config.pe_file_types` (Win32, Win64 by default).
5. Vectorize with `thrember.create_vectorized_features`, then delete all raw `.jsonl` files to reclaim disk.
6. Print disk space before/after each major step.

**Next notebook:** `02_train.ipynb` — train LightGBM, evaluate, and generate SHAP plots.

> **Note on thrember disk behaviour:**  
> `create_vectorized_features(data_dir)` reads every `.jsonl` file in `data_dir` and writes  
> `X_train.dat`, `y_train.dat`, `X_test.dat`, `y_test.dat`, `X_challenge.dat`, `y_challenge.dat`  
> back into the **same directory**.  All raw files must be present before calling it — we  
> cannot interleave download→vectorize→delete per file type.  We download everything first,  
> vectorize once, then delete all `.jsonl` files in a single pass.

## Step 1 — Install dependencies

In [ ]:
# Clone the EMBER2024 repo and install thrember (not on PyPI)
!git clone --depth 1 https://github.com/FutureComputing4AI/EMBER2024.git /tmp/EMBER2024
!pip install -q /tmp/EMBER2024

# Pin signify — 0.8+ moved SignedPEFile; thrember 0.1.0 requires the old location
!pip install -q 'signify==0.7.1'

# Install remaining LightGuard dependencies
!pip install -q lightgbm scikit-learn shap pandas numpy pyarrow \
                matplotlib seaborn watchdog pefile flask pyyaml joblib pytest

## Step 2 — Clone the LightGuard repo

**Option A (recommended):** clone from GitHub.  
**Option B:** upload the project folder as a zip, then unzip it.  
Set `REPO_URL` to your fork, or set `USE_UPLOAD = True` and skip the clone cell.

In [ ]:
# ── Edit these two lines if needed ───────────────────────────────────────────
REPO_URL  = "https://github.com/YOUR_USERNAME/LightGuard.git"  # Option A
USE_UPLOAD = False   # Set True to upload a zip instead of cloning
# ─────────────────────────────────────────────────────────────────────────────

REPO_DIR = "/content/LightGuard"

In [ ]:
import os, sys

if USE_UPLOAD:
    from google.colab import files
    print("Upload your LightGuard zip file:")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !unzip -q "{zip_name}" -d /content/
    # Rename the top-level folder to REPO_DIR if needed
    extracted = [d for d in os.listdir("/content") if os.path.isdir(f"/content/{d}") and d != "sample_data"]
    if extracted and f"/content/{extracted[0]}" != REPO_DIR:
        os.rename(f"/content/{extracted[0]}", REPO_DIR)
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"

# Make src/lightguard importable
sys.path.insert(0, f"{REPO_DIR}/src")
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Step 3 — Load config and set paths

In [ ]:
import shutil
from pathlib import Path

import yaml

with open("config.yaml") as fh:
    cfg = yaml.safe_load(fh)

# All heavy data lives on Colab's local disk — not Drive
DATA_DIR   = Path("/content/ember_data")   # raw .jsonl + vectorized .dat files
MODEL_DIR  = Path(cfg["paths"]["models"])  # relative to REPO_DIR
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PE_FILE_TYPES = cfg["pe_file_types"]        # e.g. ["Win32", "Win64"]
RANDOM_SEED   = cfg["random_seed"]

print(f"PE file types  : {PE_FILE_TYPES}")
print(f"Data directory : {DATA_DIR}")
print(f"Model directory: {MODEL_DIR.resolve()}")


def disk_free_gb(path: str = "/") -> float:
    """Return free disk space in GB at *path*."""
    stat = shutil.disk_usage(path)
    return stat.free / 1024 ** 3


def disk_used_by_gb(path: Path) -> float:
    """Return disk used by *path* and its children, in GB."""
    total = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
    return total / 1024 ** 3


print(f"\nFree disk at startup : {disk_free_gb():.1f} GB")

## Step 4 — Download EMBER2024

Downloads each PE file type from `config.pe_file_types` in turn.  
HuggingFace Hub caches downloads — if this cell is re-run, already-present files are skipped.

**Approximate sizes (compressed):**
- Win32: ~16 GB (train + test)
- Win64: ~5 GB (train + test)
- Challenge: ~1 GB (all types)

After unzipping, raw `.jsonl` files will use ~3–4× more space.  
They are deleted in Step 5 immediately after vectorization.

In [ ]:
import os
import thrember

prev_cwd = os.getcwd()

try:
    for file_type in PE_FILE_TYPES:
        print(f"\n{'='*60}")
        print(f"Downloading {file_type} (train + test) ...")
        print(f"Free disk before : {disk_free_gb():.1f} GB")

        # thrember calls os.chdir(DATA_DIR) internally; we restore cwd after.
        thrember.download_dataset(str(DATA_DIR), file_type=file_type)
        os.chdir(prev_cwd)

        used = disk_used_by_gb(DATA_DIR)
        print(f"Free disk after  : {disk_free_gb():.1f} GB")
        print(f"DATA_DIR uses    : {used:.1f} GB")

    # Challenge set is shared across file types — download once
    print(f"\n{'='*60}")
    print("Downloading challenge set ...")
    print(f"Free disk before : {disk_free_gb():.1f} GB")
    thrember.download_dataset(str(DATA_DIR), split="challenge")
    os.chdir(prev_cwd)
    print(f"Free disk after  : {disk_free_gb():.1f} GB")
    print(f"DATA_DIR uses    : {disk_used_by_gb(DATA_DIR):.1f} GB")

finally:
    os.chdir(prev_cwd)  # guarantee cwd is restored even on error

jsonl_files = list(DATA_DIR.glob("*.jsonl"))
print(f"\nTotal .jsonl files : {len(jsonl_files)}")
print(f"Free disk          : {disk_free_gb():.1f} GB")

## Step 5 — Vectorize then delete raw files

`thrember.create_vectorized_features(DATA_DIR)` reads **all** `.jsonl` files in `DATA_DIR`  
and writes six `.dat` arrays (`X_train`, `y_train`, `X_test`, `y_test`, `X_challenge`, `y_challenge`)  
into the same directory.  All raw files must be present before calling it.

After vectorization, every `.jsonl` file is deleted to reclaim disk.

In [ ]:
print(f"Free disk before vectorization : {disk_free_gb():.1f} GB")
print(f"DATA_DIR uses                  : {disk_used_by_gb(DATA_DIR):.1f} GB")
print()

thrember.create_vectorized_features(str(DATA_DIR))

print(f"\nFree disk after vectorization  : {disk_free_gb():.1f} GB")

In [ ]:
# Delete all raw .jsonl files — vectorized .dat files are all we need going forward
jsonl_files = list(DATA_DIR.glob("*.jsonl"))
print(f"Deleting {len(jsonl_files)} .jsonl files ...")
for f in jsonl_files:
    f.unlink()

remaining = list(DATA_DIR.iterdir())
print(f"Files remaining in DATA_DIR    : {len(remaining)}")
for f in sorted(remaining):
    size_gb = f.stat().st_size / 1024**3
    print(f"  {f.name:<30}  {size_gb:.2f} GB")

print(f"\nFree disk after delete         : {disk_free_gb():.1f} GB")
print(f"DATA_DIR uses                  : {disk_used_by_gb(DATA_DIR):.1f} GB")

## Step 6 — Verify arrays and row counts

In [ ]:
from lightguard.data.loaders import load_split

print(f"{'Split':<12}  {'X shape':>20}  {'y shape':>12}  {'% malicious':>12}")
print("-" * 62)

for split in ("train", "test", "challenge"):
    X, y = load_split(DATA_DIR, split)
    pct_mal = 100.0 * (y == 1).sum() / len(y)
    print(f"{split:<12}  {str(X.shape):>20}  {str(y.shape):>12}  {pct_mal:>11.1f}%")

print(f"\nFree disk : {disk_free_gb():.1f} GB")
print("\nStep 5 complete — proceed to the training notebook.")

## Step 7 — Train LightGBM + baselines

- **LightGBM:** hyperparameters from `config.yaml`; early stopping on the
  temporal validation tail carved from the training set only.
- **Baselines:** logistic regression and random forest for comparison.

The test and challenge sets are **not touched** during training.

In [ ]:
from lightguard.data.loaders import load_split, split_train_val
from lightguard.malware.train import train as train_lgbm, predict_proba

MODEL_PATH = MODEL_DIR / "lightguard_lgbm.txt"

print("Loading training data ...")
X_train_full, y_train_full = load_split(DATA_DIR, "train")
print(f"Train set : {X_train_full.shape}  ({int(y_train_full.sum()):,} malicious)")
print(f"Free disk : {disk_free_gb():.1f} GB")
print()

lgbm_model = train_lgbm(X_train_full, y_train_full, cfg, MODEL_PATH)
print(f"\nFree disk after training : {disk_free_gb():.1f} GB")

In [ ]:
from lightguard.malware.baselines import (
    train_logistic,
    train_random_forest,
    predict_proba_sklearn,
)

print("Training baselines (this may take a few minutes for RandomForest) ...")
lr_model = train_logistic(X_train_full, y_train_full, cfg, MODEL_DIR)
rf_model = train_random_forest(X_train_full, y_train_full, cfg, MODEL_DIR)
print("Baselines done.")

## Step 8 — Evaluate on test set and challenge set

**Two distinct AUC numbers are expected:**
- **Test AUC** — standard held-out test split (weeks 53-64).
- **Challenge AUC** — evasive malware mixed with test-benign rows (as per
  the official EMBER2024 `eval_lgbm.py`). This number will be lower than the
  test AUC; that is correct and expected.

Confusion matrices and ROC curves are saved to `reports/`.

In [ ]:
from lightguard.malware.evaluate import evaluate

REPORTS_DIR = Path(cfg["paths"]["reports"])
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Loading test and challenge splits ...")
X_test, y_test         = load_split(DATA_DIR, "test")
X_challenge, y_challenge = load_split(DATA_DIR, "challenge")
print(f"Test      : {X_test.shape}  ({int(y_test.sum()):,} malicious)")
print(f"Challenge : {X_challenge.shape}  ({int(y_challenge.sum()):,} malicious)")

In [ ]:
lgbm_metrics = evaluate(
    predict_fn=lambda X: predict_proba(lgbm_model, X),
    X_test=X_test,
    y_test=y_test.astype(int),
    X_challenge_raw=X_challenge,
    y_challenge_raw=y_challenge.astype(int),
    cfg=cfg,
    reports_dir=REPORTS_DIR,
    model_name="lightguard_lgbm",
)

test_auc = lgbm_metrics["lightguard_lgbm"]["test"]["roc_auc"]
chal_auc = lgbm_metrics["lightguard_lgbm"]["challenge"]["roc_auc"]
print(f"\n{'='*50}")
print(f"  LightGBM  TEST AUC      : {test_auc:.4f}")
print(f"  LightGBM  CHALLENGE AUC : {chal_auc:.4f}  (lower = expected)")
print(f"{'='*50}")

In [ ]:
for model_obj, name in [(lr_model, "baseline_lr"), (rf_model, "baseline_rf")]:
    m = evaluate(
        predict_fn=lambda X, mo=model_obj: predict_proba_sklearn(mo, X),
        X_test=X_test,
        y_test=y_test.astype(int),
        X_challenge_raw=X_challenge,
        y_challenge_raw=y_challenge.astype(int),
        cfg=cfg,
        reports_dir=REPORTS_DIR,
        model_name=name,
    )
    t = m[name]["test"]["roc_auc"]
    c = m[name]["challenge"]["roc_auc"]
    print(f"{name:<20}  TEST AUC={t:.4f}  CHALLENGE AUC={c:.4f}")

print("\nAll reports saved to:", REPORTS_DIR.resolve())

## Step 9 — Save artefacts to Google Drive

Colab sessions are ephemeral. This step mounts Drive and copies:
- `models/` — all trained model files
- `reports/` — `metrics.json` + all plot PNGs

It also creates a zip for one-click download if you prefer.

> **After this cell completes**, download `lightguard_lgbm.txt` to your laptop's
> `models/` folder and the `reports/` files to your `reports/` folder.

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")

DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/LightGuard")
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Copy models/ and reports/
for src_dir in [MODEL_DIR, REPORTS_DIR]:
    dst = DRIVE_SAVE_DIR / src_dir.name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src_dir, dst)
    print(f"Saved {src_dir.name}/ → {dst}")

print(f"\nDrive save complete: {DRIVE_SAVE_DIR}")

In [ ]:
# Also zip for direct download from the Files panel
zip_base = "/content/lightguard_artefacts"
zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=REPO_DIR,
    base_dir=".",  # include models/ and reports/ from repo root
)
print(f"Zip created: {zip_path}")
print("Download it from the Colab Files panel (left sidebar → folder icon).")

# Final summary
import json as _json
all_metrics = _json.loads((REPORTS_DIR / "metrics.json").read_text())
print("\n" + "="*55)
print(f"{'Model':<22}  {'Test AUC':>10}  {'Challenge AUC':>14}")
print("-"*55)
for mname, splits in all_metrics.items():
    t = splits["test"]["roc_auc"]
    c = splits["challenge"]["roc_auc"]
    print(f"{mname:<22}  {t:>10.4f}  {c:>14.4f}")
print("="*55)

## Step 10 — SHAP explainability

Compute SHAP values for a sample of the test set and generate:
1. **Global summary plot** — which features matter most across the whole population.
2. **Two waterfall plots** — one benign sample, one malicious sample.

All plots are saved to `reports/` and synced to Drive.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import shap

from lightguard.explain.explainer import build_explainer, explain_prediction
from lightguard.explain.translate import FEATURE_NAMES, translate

# ── Background sample: 500 random rows from the test set ─────────────────────
SHAP_BACKGROUND_N = 500
SHAP_SAMPLE_N     = 1000  # rows used for global summary

rng = np.random.default_rng(cfg["random_seed"])

bg_idx     = rng.choice(len(X_test), size=SHAP_BACKGROUND_N, replace=False)
sample_idx = rng.choice(len(X_test), size=SHAP_SAMPLE_N,     replace=False)

X_bg     = X_test[bg_idx]
X_sample = X_test[sample_idx]
y_sample = y_test[sample_idx]

print(f"Background : {X_bg.shape}")
print(f"SHAP sample: {X_sample.shape}  ({int((y_sample==1).sum())} malicious)")

explainer = build_explainer(lgbm_model, X_bg)
print("TreeExplainer built.")

In [ ]:
# ── Global SHAP summary plot ──────────────────────────────────────────────────
print(f"Computing SHAP values for {SHAP_SAMPLE_N} samples (may take a few minutes) ...")
shap_values = explainer.shap_values(X_sample)  # shape (n_samples, n_features)

# Show only the top-30 features to keep the plot readable
TOP_N = 30
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_feat_idx  = np.argsort(mean_abs_shap)[::-1][:TOP_N]

X_top    = X_sample[:, top_feat_idx]
sv_top   = shap_values[:, top_feat_idx]
names_top = [FEATURE_NAMES[i] for i in top_feat_idx]

fig, ax = plt.subplots(figsize=(10, 9))
shap.summary_plot(sv_top, X_top, feature_names=names_top, show=False, plot_size=None)
plt.title("SHAP Summary — Top 30 features (LightGBM, 1,000 test samples)", fontsize=12)
plt.tight_layout()

summary_path = REPORTS_DIR / "shap_summary_lgbm.png"
fig.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved → {summary_path}")

print("\nTop 10 features by mean |SHAP|:")
for rank, i in enumerate(top_feat_idx[:10], 1):
    print(f"  {rank:>2}. {FEATURE_NAMES[i]:<45}  mean|SHAP|={mean_abs_shap[i]:.4f}")

In [ ]:
# ── Per-sample waterfall plots (one benign, one malicious) ────────────────────
from lightguard.malware.train import predict_proba as lgbm_predict

# Pick the first benign and first malicious sample from the SHAP sample
benign_rows    = np.where(y_sample == 0)[0]
malicious_rows = np.where(y_sample == 1)[0]

assert len(benign_rows) > 0,    "No benign samples in SHAP sample — re-run with a larger SHAP_SAMPLE_N"
assert len(malicious_rows) > 0, "No malicious samples in SHAP sample"

for label, row_idx in [("benign", benign_rows[0]), ("malicious", malicious_rows[0])]:
    x         = X_sample[row_idx]
    sv        = shap_values[row_idx]
    score     = lgbm_predict(lgbm_model, x.reshape(1, -1))[0]

    # Top-15 features for the waterfall
    top15_idx  = np.argsort(np.abs(sv))[::-1][:15]
    top15_sv   = sv[top15_idx]
    top15_x    = x[top15_idx]
    top15_names = [FEATURE_NAMES[i] for i in top15_idx]

    # Build a shap.Explanation for the waterfall plot
    base_value = explainer.expected_value
    if isinstance(base_value, (list, np.ndarray)):
        base_value = float(base_value[0])

    explanation = shap.Explanation(
        values=top15_sv,
        base_values=base_value,
        data=top15_x,
        feature_names=top15_names,
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    shap.waterfall_plot(explanation, max_display=15, show=False)
    plt.title(
        f"SHAP Waterfall — {label.capitalize()} sample  "
        f"(malware score={score:.3f})",
        fontsize=12,
    )
    plt.tight_layout()

    path = REPORTS_DIR / f"shap_waterfall_{label}_lgbm.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved → {path}")

    print(f"\nPlain-English explanation ({label}):")
    top5 = explain_prediction(explainer, x, top_k=5)
    for sent in translate(top5):
        print(f"  • {sent}")

## Step 11 — Export test holdout sample

Export ~2,000 random rows from the test set to `data/sample/` so the laptop has:
- A realistic background distribution for the SHAP explainer at inference time.
- A small labelled test set for local demos and smoke tests.

These files (`test_holdout_X.npy`, `test_holdout_y.npy`) are committed to git and travel with the repo.

In [ ]:
HOLDOUT_N    = 2000
HOLDOUT_SEED = cfg["random_seed"]

rng_holdout = np.random.default_rng(HOLDOUT_SEED)
holdout_idx = rng_holdout.choice(len(X_test), size=HOLDOUT_N, replace=False)

X_holdout = X_test[holdout_idx]
y_holdout = y_test[holdout_idx].astype(np.float32)

sample_out = Path(cfg["paths"]["data_sample"])
sample_out.mkdir(parents=True, exist_ok=True)

X_path = sample_out / "test_holdout_X.npy"
y_path = sample_out / "test_holdout_y.npy"

np.save(X_path, X_holdout)
np.save(y_path, y_holdout)

x_mb = X_path.stat().st_size / 1024**2
y_mb = y_path.stat().st_size / 1024**2

print(f"Exported holdout sample:")
print(f"  X shape : {X_holdout.shape}   →  {X_path}  ({x_mb:.1f} MB)")
print(f"  y shape : {y_holdout.shape}   →  {y_path}  ({y_mb:.2f} MB)")
print(f"  {int((y_holdout==1).sum())} malicious / {int((y_holdout==0).sum())} benign")
print(f"\nCommit these files to git and push — they travel to the laptop.")
print("  git add data/sample/test_holdout_X.npy data/sample/test_holdout_y.npy")
print("  git commit -m 'data: add test holdout sample for local SHAP inference'")

# Also copy to Drive
for p in [X_path, y_path]:
    dst = DRIVE_SAVE_DIR / "data_sample" / p.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dst)
    print(f"Backed up → {dst}")

In [ ]:
# Sync the new SHAP reports to Drive
for src_dir in [MODEL_DIR, REPORTS_DIR]:
    dst = DRIVE_SAVE_DIR / src_dir.name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src_dir, dst)
    print(f"Synced {src_dir.name}/ → {dst}")

# Final artefact list
print("\nArtefacts ready on Drive:")
for f in sorted(DRIVE_SAVE_DIR.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1024**2
        print(f"  {str(f.relative_to(DRIVE_SAVE_DIR)):<55}  {size_mb:>7.2f} MB")

print("\nM3 complete.")
print("Download to laptop:")
print("  models/lightguard_lgbm.txt   →  models/")
print("  reports/*                    →  reports/")
print("  data_sample/test_holdout_*.npy → data/sample/")